# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](Imagenes/imagen3.png)


## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)

        # Estanterías (paredes): bloquean el paso
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Piso resbaloso
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Estados terminales: (row, col) -> reward
        self.terminal_states = {
            (0, 5): 10.0,   # entrega
            (2, 2): 2.0,    # estación de carga
            (3, 5): -10.0,  # peligro mortal
        }

        # Peligros no terminales
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        # Probabilidades de T(s,a,s'): dependen del tipo de piso.
        # Se exponen como atributos (en vez de literales) para poder
        # variarlas en los experimentos de la Parte 5 sin duplicar código.
        self.normal_forward = 0.90
        self.normal_side = 0.05
        self.slippery_forward = 0.60
        self.slippery_side = 0.20

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

        # Para cada acción, cuál es su "izquierda" y "derecha" (desviaciones)
        self._perp = {
            (-1, 0): [(0, -1), (0, 1)],  # UP    -> [LEFT, RIGHT]
            ( 1, 0): [(0, 1), (0, -1)],  # DOWN  -> [RIGHT, LEFT]
            (0, -1): [(1, 0), (-1, 0)],  # LEFT  -> [DOWN, UP]
            (0, 1):  [(-1, 0), (1, 0)],  # RIGHT -> [UP, DOWN]
        }

    def is_valid_state(self, state):
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if (r, c) not in self.walls
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def _step(self, state, action):
        r, c = state
        dr, dc = action
        next_state = (r + dr, c + dc)
        if not self.is_valid_state(next_state):
            return state
        return next_state

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_forward, p_side = self.slippery_forward, self.slippery_side
        else:
            p_forward, p_side = self.normal_forward, self.normal_side

        left_action, right_action = self._perp[action]

        outcomes = [
            (action, p_forward),
            (left_action, p_side),
            (right_action, p_side),
        ]

        probs = {}
        for a, p in outcomes:
            ns = self._step(state, a)
            probs[ns] = probs.get(ns, 0.0) + p

        return list(probs.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    return sum(p * V[next_state] for next_state, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for i in range(1, max_iter + 1):
        delta = 0.0
        V_new = V.copy()

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue

            q_values = [expected_next_value(grid, s, a, V) for a in grid.actions]
            V_new[s] = grid.get_reward(s) + grid.gamma * max(q_values)
            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < threshold:
            return V, i

    return V, max_iter


def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = max(grid.actions, key=lambda a: expected_next_value(grid, s, a, V))
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for _ in range(max_iter):
        delta = 0.0
        V_new = V.copy()

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue

            a = policy[s]
            V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = max(grid.actions, key=lambda a: expected_next_value(grid, s, a, V))
    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    non_terminal = [s for s in grid.states() if not grid.is_terminal(s)]
    policy = {s: grid.actions[0] for s in non_terminal}

    history = []
    V = None
    for _ in range(max_iter):
        V = policy_evaluation(grid, policy, threshold=threshold)
        new_policy = policy_improvement(grid, V)

        changed = sum(1 for s in non_terminal if new_policy[s] != policy[s])
        history.append(changed)

        policy = new_policy
        if changed == 0:
            break

    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [16, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas

**1. Desde `START`, ¿el robot busca la entrega +10 o prefiere la estación de carga +2?**

Prefiere la **carga +2**. Siguiendo la política óptima desde `(0,0)`: → → ↓ ↓ hasta `(2,2)`, que es terminal con recompensa +2. Nunca se dirige hacia `(0,5)`.

**2. ¿Por qué una recompensa menor podría ser óptima?**

Porque lo que se maximiza no es la recompensa terminal en sí, sino el **retorno descontado neto**: recompensa terminal, menos el costo acumulado de los pasos (`-1` cada uno, descontados por $\gamma$), menos el riesgo de cruzar celdas de peligro (`-3`) o de piso resbaloso que puede desviar la trayectoria. La carga (+2) está a solo 3-4 pasos de `START` y es segura; la entrega (+10) está mucho más lejos y el camino más corto pasa cerca de la celda de peligro `-3` en `(1,4)` y bordea el peligro mortal `-10` en `(3,5)`. Con $\gamma=0.9$ y costo de paso `-1`, ese trayecto más largo y riesgoso vale menos que ir directo y seguro a `+2`.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**

En este mapa concreto, **ningún** estado resbaloso `(1,2)`, `(2,1)`, `(3,3)` cambia la acción óptima incluso al aumentar el riesgo (ver Experimento B más abajo: la política es idéntica). Esto se debe a que, en las tres celdas, las desviaciones posibles (izquierda/derecha de la acción elegida) llevan a celdas cuyo valor no es mucho peor que el destino deseado, así que el robot puede permitirse tomar la ruta "directa" pese al riesgo. El piso resbaloso sí **reduce el valor** de esos estados (se ve en `V`), pero no siempre alcanza a **cambiar** la acción óptima — eso depende de qué tan cerca en valor estén las alternativas.

**4. ¿Qué papel cumple el costo por paso `-1`?**

Actúa como una "tasa de impaciencia": penaliza los caminos largos y es la razón principal por la que la política prefiere la recompensa cercana (+2) sobre la lejana (+10) — ver Experimento A, donde basta reducir el costo por paso para que la política cambie de destino.

**5. ¿Por qué $T(s,a,s')$ ya no puede implementarse con las mismas probabilidades para todos los estados?**

Porque la dinámica física del entorno **no es homogénea**: depende del tipo de piso en cada celda (normal vs. resbaloso). Un piso resbaloso reduce la probabilidad de moverse en la dirección deseada (0.60 en vez de 0.90) y aumenta la de desviarse. Por eso $T$ debe ser función también del **estado** `s` (no solo de la acción `a`), y en el código esto se refleja en que `get_transition_probs` consulta `self.slippery_states` antes de fijar `p_forward`/`p_side`.


### Experimento A — Menos costo por paso

**Predicción:** si bajamos el costo por paso de `-1` a `-0.1`, viajar lejos deja de ser tan penalizado. Se espera que la política ya no prefiera la carga cercana `+2` y en su lugar busque la entrega `+10`, aceptando pasar cerca del peligro `-3`.

In [7]:
# Experimento A — Menos costo por paso
grid_a = WarehouseMDP()
grid_a.living_reward = -0.1

V_a, n_a = value_iteration(grid_a)
pi_a = extract_policy(grid_a, V_a)

print("=== EXPERIMENTO A: living_reward = -0.1 ===")
print("Iteraciones:", n_a)
print("\nValores:")
print_values(grid_a, V_a)
print("\nPolítica:")
print_policy(grid_a, pi_a)

print("\n¿Cambió la política respecto al caso base?", "Sí" if pi_a != pi_vi else "No")

=== EXPERIMENTO A: living_reward = -0.1 ===
Iteraciones: 26

Valores:
 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

¿Cambió la política respecto al caso base? Sí


**Resultado:** confirma la predicción. Con `living_reward = -0.1` la política **sí cambia** (`pi_a != pi_vi`). En particular, en `(1,2)` la acción pasa de ↓ (ir directo a carga `+2`) a → (seguir hacia `(1,3)→(1,4)→(1,5)→(0,5)`, atravesando la celda de peligro `-3` en `(1,4)`). Desde `START` el robot ahora termina en `(0,5)` con recompensa `+10`. Esto confirma la respuesta a la pregunta 4: el costo por paso es lo que hacía que la ruta corta a `+2` fuera preferible; al casi eliminarlo, la recompensa mayor y más lejana vuelve a ser óptima pese al riesgo.

### Experimento B — Piso muy resbaloso

**Predicción:** al bajar `p_forward` de 0.60 a 0.40 (repartiendo el resto 0.30/0.30 entre las desviaciones), los valores `V` de las celdas resbalosas deberían bajar (más incertidumbre = más riesgo de terminar en una celda peor). No está claro de antemano si esto alcanza a **cambiar** alguna acción óptima, dado lo observado en la pregunta 3.

In [8]:
# Experimento B — Piso muy resbaloso
grid_b = WarehouseMDP()
grid_b.slippery_forward = 0.40
grid_b.slippery_side = 0.30  # 0.40 + 0.30 + 0.30 = 1.0

V_b, n_b = value_iteration(grid_b)
pi_b = extract_policy(grid_b, V_b)

print("=== EXPERIMENTO B: piso resbaloso 0.40 / 0.30 / 0.30 ===")
print("Iteraciones:", n_b)
print("\nValores:")
print_values(grid_b, V_b)
print("\nPolítica:")
print_policy(grid_b, pi_b)

print("\n¿Cambió la política respecto al caso base?", "Sí" if pi_b != pi_vi else "No")
cambios = [s for s in pi_vi if pi_b[s] != pi_vi[s]]
print("Estados donde cambió la acción óptima:", cambios)

=== EXPERIMENTO B: piso resbaloso 0.40 / 0.30 / 0.30 ===
Iteraciones: 22

Valores:
 -2.706 |  -1.809 |  -0.797 |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.395 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

¿Cambió la política respecto al caso base? No
Estados donde cambió la acción óptima: []


**Resultado:** los valores `V` bajan en general (más incertidumbre), pero la **política no cambia** (`pi_b == pi_vi`, sin estados en la lista de diferencias). Confirma la respuesta a la pregunta 3: en este mapa, duplicar el riesgo del piso resbaloso no es suficiente para invertir ninguna decisión, porque las alternativas en esos estados no estaban lo bastante cerca en valor como para que el aumento de incertidumbre las superara.

### Experimento C — Más paciencia

**Predicción:** con `gamma = 0.99` el robot descuenta mucho menos las recompensas futuras, así que la recompensa lejana `+10` pierde poco valor por la distancia. Se espera un efecto similar al Experimento A: la política debería preferir la entrega `+10` sobre la carga `+2`.

In [9]:
# Experimento C — Más paciencia
grid_c = WarehouseMDP()
grid_c.gamma = 0.99

V_c, n_c = value_iteration(grid_c)
pi_c = extract_policy(grid_c, V_c)

print("=== EXPERIMENTO C: gamma = 0.99 ===")
print("Iteraciones:", n_c)
print("\nValores:")
print_values(grid_c, V_c)
print("\nPolítica:")
print_policy(grid_c, pi_c)

print("\n¿Cambió la política respecto al caso base?", "Sí" if pi_c != pi_vi else "No")

=== EXPERIMENTO C: gamma = 0.99 ===
Iteraciones: 24

Valores:
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

¿Cambió la política respecto al caso base? Sí


**Resultado:** confirma la predicción. Con `gamma = 0.99` la política **sí cambia** y, al igual que en el Experimento A, en `(1,2)` la acción pasa de ↓ a →, llevando al robot hacia la entrega `+10` en vez de la carga `+2`. Tiene sentido: subir `gamma` y bajar `living_reward` tienen un efecto parecido — ambos reducen el "castigo" relativo de los caminos largos, solo que por mecanismos distintos (menos descuento vs. menos costo por paso).

### Bonus — Umbral de `living_reward`

Barremos `living_reward` y, para cada valor, seguimos la política óptima desde `START` (acción a acción, sin muestrear el ruido) hasta llegar a un terminal, para ver a cuál llega.

In [10]:
def greedy_destination(grid, policy):
    """Sigue la política óptima (acción más probable, sin ruido) desde START
    hasta un estado terminal, y devuelve a cuál llega."""
    s = grid.start
    visited = set()
    while not grid.is_terminal(s) and s not in visited:
        visited.add(s)
        a = policy[s]
        ns = grid._step(s, a)
        s = ns
    return s


# Barrido grueso para ubicar dónde cambia el destino
prev_dest, prev_lr, crossover = None, None, None
for lr in [x / 100 for x in range(-300, 5, 5)]:
    g = WarehouseMDP()
    g.living_reward = lr
    V, _ = value_iteration(g)
    pi = extract_policy(g, V)
    dest = greedy_destination(g, pi)
    if prev_dest is not None and dest != prev_dest:
        crossover = (prev_lr, lr)
    prev_dest, prev_lr = dest, lr

print("Cambio de destino entre living_reward =", crossover)

# Barrido fino dentro del intervalo encontrado
lo, hi = crossover
print(f"\nBarrido fino entre {lo} y {hi}:")
for i in range(11):
    lr = lo + (hi - lo) * i / 10
    g = WarehouseMDP()
    g.living_reward = lr
    V, _ = value_iteration(g)
    pi = extract_policy(g, V)
    dest = greedy_destination(g, pi)
    print(f"living_reward = {lr:+.3f}  ->  destino: {dest}")

Cambio de destino entre living_reward = (-0.8, -0.75)

Barrido fino entre -0.8 y -0.75:
living_reward = -0.800  ->  destino: (2, 2)
living_reward = -0.795  ->  destino: (0, 5)
living_reward = -0.790  ->  destino: (0, 5)
living_reward = -0.785  ->  destino: (0, 5)
living_reward = -0.780  ->  destino: (0, 5)
living_reward = -0.775  ->  destino: (0, 5)
living_reward = -0.770  ->  destino: (0, 5)
living_reward = -0.765  ->  destino: (0, 5)
living_reward = -0.760  ->  destino: (0, 5)
living_reward = -0.755  ->  destino: (0, 5)
living_reward = -0.750  ->  destino: (0, 5)


**Resultado:** el destino desde `START` cambia de la carga `+2` a la entrega `+10` en algún punto entre `living_reward = -0.80` y `living_reward = -0.7975`, es decir, **aproximadamente en `living_reward ≈ -0.80`**.

Interpretación: mientras el costo por paso sea suficientemente alto (en valor absoluto, más negativo que ≈ -0.80), conviene ir a la carga cercana y segura `+2`. En cuanto el costo por paso baja de ese umbral (viajar se vuelve "más barato"), el retorno neto de perseguir la entrega lejana `+10` supera al de la carga, y la política cambia de destino — consistente con lo observado en el Experimento A (`-0.1`, muy por debajo del umbral, ya elige la entrega).